# 🌬️ 따라쓰기 실습. 전국 평균풍속 지도 그리기

**AI로 분석하는 신재생에너지 · 울산대학교**

기상청의 실제 풍속 데이터(2016.7 ~ 2021.6 · 5년 평균 · 고도 80m)로

**"왜 울산 앞바다인가?"** 를 데이터로 직접 확인합니다.

### 📌 이 파일 사용법

1. 각 단계의 **회색 상자 안 코드**를 아래 빈 셀에 **직접 타이핑**합니다
2. `Shift + Enter` 를 눌러 실행합니다
3. 에러가 나면? **에러 메시지 마지막 줄을 읽어보세요**

> 마우스로 4개 지점을 20분 걸려 조회했던 일을, 코드는 **전국 4,657만 개 지점을 몇 초에** 해냅니다.

---
## 1단계. 준비 — 도구 설치

```python
!pip install netcdf4    # nc 파일을 읽기 위한 도구 설치 (!는 '파이썬 밖 명령'이라는 뜻)
```

💬 코랩에 기본으로 없는 도구는 이렇게 **pip으로 설치**해서 씁니다. 아까 배운 그 방법입니다.

In [ ]:
# ✏️ 위 코드를 여기에 따라 입력하세요


---
## 2단계. 데이터 파일 올리고 확인하기

**먼저 파일을 올립니다** — `KMAPP_wind_total_mean.nc` (약 186MB, 커서 시간이 좀 걸립니다)

- 왼쪽 📁 파일 탭을 열고, nc 파일을 **드래그해서 놓기**
- ⚠️ [파일 업로드] 버튼의 선택 창에 nc 파일이 안 보일 수 있습니다 → **드래그앤드롭**으로 하세요
- 업로드 중 표시(주황색 원)가 끝날 때까지 기다린 뒤 아래를 실행합니다

```python
import os                        # 파일과 폴더를 다루는 기본 도구

print(os.listdir('/content'))    # 코랩 작업 폴더에 있는 파일 목록 출력 — 업로드 확인용
```

💬 목록에 `KMAPP_wind_total_mean.nc` 가 보이면 준비 완료입니다.

In [ ]:
# ✏️ 위 코드를 여기에 따라 입력하세요


---
## 3단계. nc 파일 열어보기

```python
import xarray as xr                                       # 큐브형 데이터를 다루는 도구, 별명은 xr

ds = xr.open_dataset('/content/KMAPP_wind_total_mean.nc') # nc 파일을 열어서 ds라는 이름으로 저장

print(ds)                                                 # 안에 뭐가 들었는지 요약 출력
```

### 💡 nc 파일이란?

- **엑셀·CSV는 표**, nc는 **큐브** — 공간(가로×세로)이나 시간까지 한 파일에 담는 과학 데이터 형식
- 기상청·해양·위성 데이터가 대부분 이 형식입니다

**✅ 출력에서 확인할 것**
- 격자 크기: `6900 × 6750` → 곱하면 **약 4,657만 칸** (한 칸 = 가로세로 100m)
- 변수 이름: `windspeed` 하나

In [ ]:
# ✏️ 위 코드를 여기에 따라 입력하세요


---
## 4단계. 전국 풍속 지도 그리기

```python
import matplotlib.pyplot as plt          # 그래프를 그리는 도구, 별명은 plt

ds['windspeed'].plot(figsize=(9, 9))     # 풍속 데이터를 색깔 지도로 그리기 (figsize는 그림 크기)

plt.show()                               # 그림을 화면에 표시
```

### 🤔 그림이 나오면 생각해 보기

**우리는 해안선을 그린 적이 없습니다. 그런데 한반도 모양이 보입니다. 왜일까요?**

→ 풍속 숫자만으로 땅과 바다가 구분된다는 것 — 이것이 데이터가 지형을 말하는 방식입니다.

In [ ]:
# ✏️ 위 코드를 여기에 따라 입력하세요


---
## 5단계. 전국 통계 한 번에 계산하기

```python
import numpy as np                                # 숫자 계산 도구, 별명은 np

ws = ds['windspeed'].values                       # 풍속 값만 꺼내서 numpy 배열로 저장

print('최대 풍속:', round(np.nanmax(ws), 2), 'm/s')    # 전국에서 가장 센 곳 (nan = 빈 값 무시)
print('평균 풍속:', round(np.nanmean(ws), 2), 'm/s')   # 전국 평균
```

💬 `nanmax` 처럼 **nan이 붙은 함수**는 빈 값(NaN)을 건너뛰고 계산합니다 — 실제 데이터엔 빈 값이 있으니까요.

**✅ 확인** — 최대 약 `8.47 m/s`, 평균 약 `6.10 m/s` 가 나오면 정상입니다.

마우스로 4곳 조회한 것과 비교해 보세요. **전국 4,657만 지점의 통계가 방금 몇 초에 끝났습니다.**

In [ ]:
# ✏️ 위 코드를 여기에 따라 입력하세요


---
## 6단계. 사업성 기준선 8 m/s — 얼마나 되나?

해상풍력이 돈이 되려면 평균풍속 **8 m/s** 는 나와야 한다고 했습니다(Day 1).

그런 땅과 바다가 우리나라에 얼마나 있을까요?

```python
count = np.nansum(ws >= 8.0)              # 8 이상인 칸의 개수 (참=1, 거짓=0으로 더해짐)

ratio = count / ws.size * 100             # 전체 칸 대비 비율(%) 계산

area = count * 0.01                       # 면적으로 환산 — 한 칸이 100m×100m = 0.01 km²

print('8 m/s 이상 칸 수:', count)                        # 개수 출력
print('전체의', round(ratio, 1), '%')                    # 비율 출력
print('면적으로 약', format(int(area), ','), 'km²')      # 면적 출력 (천 단위 콤마)
```

**✅ 확인** — 약 `6.4 %`, 약 `29,700 km²` 가 나오면 정상입니다.

💬 29,700km² = **울산광역시 면적의 약 28배.** 좁지 않죠. 문제는 그게 **어디에** 있느냐입니다.

**🎯 미니 챌린지** — 기준을 `7.0` 으로 바꾸면 비율이 몇 %가 될까요? 위 코드의 8.0만 고쳐서 직접 확인해 보세요. (7 m/s는 부유식 해상풍력이 가능하다고 보는 기준입니다)

In [ ]:
# ✏️ 위 코드를 여기에 따라 입력하세요


---
## ✅ 오늘 쓴 함수 정리

| 함수 | 하는 일 |
|---|---|
| `!pip install 이름` | 코랩에 없는 도구 설치 |
| `xr.open_dataset(경로)` | nc(큐브형) 파일 열기 |
| `ds['변수'].plot()` | 데이터를 색깔 지도로 그리기 |
| `np.nanmax / np.nanmean` | 빈 값 무시하고 최대·평균 |
| `np.nansum(조건)` | 조건에 맞는 칸 개수 세기 |

### 📌 오늘의 결론

- 마우스 4곳 20분 → **코드 4,657만 곳 몇 초**
- 전국 최대 **8.47 m/s** · 평균 **6.10 m/s**
- 8 m/s 이상은 전체의 **6.4%** (약 29,700 km² — 울산시 면적의 약 28배)
- 해안선을 그린 적 없는데 한반도가 보였다 — **데이터가 지형을 말한다**

어디에 지을지는 데이터가 알려줬습니다. 이제 **어떻게 돌릴 것인가**(Q2)의 문제입니다.